In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

from dotenv import load_dotenv

load_dotenv()

CF_OUTPUTS = Path(os.getenv("CF_OUTPUTS"))
print(CF_OUTPUTS)
print(CF_OUTPUTS.is_dir())


/home/dyretna/Dokument/Code/GitHub/nightingale_projects/cf_bench/cf_outputs
True


In [ ]:
batch_cfcheck_data = CF_OUTPUTS / "RandomForest_random__highthres_2026-04-21" / "random_annotated_hltprhc.csv"
batch_cfcheck_df = pd.read_csv(batch_cfcheck_data)

In [3]:
pd.set_option("display.max_rows", None)

In [4]:
batch_cfcheck_df = batch_cfcheck_df.drop(columns=["hltprhc", "target_risk"])

In [5]:

batch_cfcheck_df["cf_id"] = batch_cfcheck_df["cf_id"].replace({"original": "xin"})

batch_cfcheck_df = batch_cfcheck_df.rename(columns={
    "original_risk": "risk_before",
    "predicted_risk" : "predicted_risk_after",
    "meets_target_risk" : "valid",
})

batch_cfcheck_df.head(4)

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,risk_before,predicted_risk_after,valid,cf_gen_time_sec
0,0,xin,4.0,3.0,3.0,4.0,2.0,0.0,18.986591,0.0,0.063333,NaN,NaN,0.34
1,0,cf_1,NaN,NaN,NaN,NaN,NaN,NaN,17.134570,NaN,0.063333,0.046667,False,NaN
2,0,cf_2,NaN,NaN,NaN,NaN,NaN,NaN,30.168300,NaN,0.063333,0.113333,False,NaN
3,0,cf_3,7.0,NaN,NaN,NaN,NaN,NaN,16.951940,NaN,0.063333,0.083333,False,NaN


In [6]:
int_columns = [
    "etfruit",
    "eatveg",
    "cgtsmok",
    "alcfreq",
    "slprl",
    "paccnois",
    "dosprt",
]

float_columns = [
    "bmi",
    "risk_before",
    "predicted_risk_after",
]

In [7]:
batch_cfcheck_df[int_columns] = batch_cfcheck_df[int_columns].astype("Int64")

In [8]:
batch_cfcheck_df[float_columns] = batch_cfcheck_df[float_columns].round(2)

batch_cfcheck_df.head(4)

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,risk_before,predicted_risk_after,valid,cf_gen_time_sec
0,0,xin,4,3,3,4,2,0,18.99,0,0.06,NaN,NaN,0.34
1,0,cf_1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,17.13,<NA>,0.06,0.05,False,NaN
2,0,cf_2,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,30.17,<NA>,0.06,0.11,False,NaN
3,0,cf_3,7,<NA>,<NA>,<NA>,<NA>,<NA>,16.95,<NA>,0.06,0.08,False,NaN


In [9]:
batch_cfcheck_df[int_columns] = (
    batch_cfcheck_df[int_columns]
    .astype("string")
    .fillna("")
)

In [10]:
batch_cfcheck_df = batch_cfcheck_df.replace({np.nan : ""})
batch_cfcheck_df.head(4)

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,risk_before,predicted_risk_after,valid,cf_gen_time_sec
0,0,xin,4,3,3,4,2,0,18.99,0,0.06,,,0.34
1,0,cf_1,,,,,,,17.13,,0.06,0.05,False,
2,0,cf_2,,,,,,,30.17,,0.06,0.11,False,
3,0,cf_3,7,,,,,,16.95,,0.06,0.08,False,


In [11]:
mask = batch_cfcheck_df["cf_id"] == "xin"
batch_cfcheck_df.loc[mask, "risk_before"] = ""
batch_cfcheck_df.head(4)

/tmp/ipykernel_10590/1160248817.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  batch_cfcheck_df.loc[mask, "risk_before"] = ""


,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,risk_before,predicted_risk_after,valid,cf_gen_time_sec
0,0,xin,4,3,3,4,2,0,18.99,0,,,,0.34
1,0,cf_1,,,,,,,17.13,,0.06,0.05,False,
2,0,cf_2,,,,,,,30.17,,0.06,0.11,False,
3,0,cf_3,7,,,,,,16.95,,0.06,0.08,False,


In [12]:
feature_cols = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "bmi", "dosprt"]

# 1. Beräkna Nchanged endast för CF-rader
mask_cf = batch_cfcheck_df["cf_id"] != "xin"

batch_cfcheck_df.loc[mask_cf, "Nchanged"] = (
    batch_cfcheck_df.loc[mask_cf, feature_cols] != ""
).sum(axis=1)

# 2. Konvertera kolumnen till ren string
batch_cfcheck_df["Nchanged"] = batch_cfcheck_df["Nchanged"].astype("string")

# 3. Baseline ska vara tom
batch_cfcheck_df.loc[batch_cfcheck_df["cf_id"] == "xin", "Nchanged"] = ""

In [13]:
batch_cfcheck_df.head(4)

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,risk_before,predicted_risk_after,valid,cf_gen_time_sec,Nchanged
0,0,xin,4,3,3,4,2,0,18.99,0,,,,0.34,
1,0,cf_1,,,,,,,17.13,,0.06,0.05,False,,1
2,0,cf_2,,,,,,,30.17,,0.06,0.11,False,,1
3,0,cf_3,7,,,,,,16.95,,0.06,0.08,False,,2


In [14]:
batch_cfcheck_df["Gower"] = ""
batch_cfcheck_df["Expected"] = ""


In [15]:
order = ["query_index", "cf_id"] + feature_cols + ["cf_gen_time_sec", "Gower", "Nchanged", "valid", "risk_before", "predicted_risk_after", "Expected"]

batch_cfcheck_df = batch_cfcheck_df[order]
batch_cfcheck_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,Gower,Nchanged,valid,risk_before,predicted_risk_after,Expected
0,0,xin,4,3,3,4,2,0,18.99,0,0.34,,,,,,
1,0,cf_1,,,,,,,17.13,,,,1,False,0.06,0.05,
2,0,cf_2,,,,,,,30.17,,,,1,False,0.06,0.11,
3,0,cf_3,7,,,,,,16.95,,,,2,False,0.06,0.08,
4,0,cf_4,,6,,,,,27.88,,,,2,False,0.06,0.2,
5,0,cf_5,,4,1,,,,,,,,2,True,0.06,0.03,
6,0,cf_6,,2,,,,1,,,,,2,False,0.06,0.11,
7,0,cf_7,,,,,,1,37.07,,,,2,False,0.06,0.09,
8,0,cf_8,5,,,,,,29.89,,,,2,False,0.06,0.06,
9,0,cf_9,,,,,,,33.65,,,,1,False,0.06,0.22,


# picking correct CF

### expected


In [16]:
# Check directional constraints directly
# These features should only improve (increase) or stay the same
SHOULD_INCREASE = ["cgtsmok", "alcfreq", "dosprt"]

# These features should only improve (decrease) or stay the same
SHOULD_DECREASE = ["bmi", "etfruit", "eatveg", "slprl", "paccnois"]


print("Directional constraints:")
print(f"  Should increase (↑): {SHOULD_INCREASE}")
print(f"  Should decrease (↓): {SHOULD_DECREASE}")


Directional constraints:
  Should increase (↑): ['cgtsmok', 'alcfreq', 'dosprt']
  Should decrease (↓): ['bmi', 'etfruit', 'eatveg', 'slprl', 'paccnois']


In [17]:
# Check if CFs violate directional constraints
# First, ensure numeric columns are actually numeric for comparison
numeric_cols = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "bmi", "dosprt"]

# Create a working copy with numeric values
df_check = batch_cfcheck_df.copy()
for col in numeric_cols:
    df_check[col] = pd.to_numeric(df_check[col], errors="coerce")

# Extract baseline values per query_index
baseline_values = df_check[df_check["cf_id"] == "xin"].set_index("query_index")[numeric_cols]

# Check violations for each CF
def check_violations(row):
    if row["cf_id"] == "xin":
        return ""

    baseline = baseline_values.loc[row["query_index"]]
    violations = []

    # Check features that should increase
    for feat in SHOULD_INCREASE:
        if pd.notna(row[feat]) and pd.notna(baseline[feat]):
            if row[feat] < baseline[feat]:
                violations.append(f"{feat}↓")

    # Check features that should decrease
    for feat in SHOULD_DECREASE:
        if pd.notna(row[feat]) and pd.notna(baseline[feat]):
            if row[feat] > baseline[feat]:
                violations.append(f"{feat}↑")

    return "NO: " + ", ".join(violations) if violations else ""

batch_cfcheck_df["Expected"] = df_check.apply(check_violations, axis=1)
batch_cfcheck_df.head(10)


,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,Gower,Nchanged,valid,risk_before,predicted_risk_after,Expected
0,0,xin,4,3,3,4,2,0,18.99,0,0.34,,,,,,
1,0,cf_1,,,,,,,17.13,,,,1,False,0.06,0.05,
2,0,cf_2,,,,,,,30.17,,,,1,False,0.06,0.11,NO: bmi↑
3,0,cf_3,7,,,,,,16.95,,,,2,False,0.06,0.08,NO: etfruit↑
4,0,cf_4,,6,,,,,27.88,,,,2,False,0.06,0.2,"NO: bmi↑, eatveg↑"
5,0,cf_5,,4,1,,,,,,,,2,True,0.06,0.03,"NO: cgtsmok↓, eatveg↑"
6,0,cf_6,,2,,,,1,,,,,2,False,0.06,0.11,NO: paccnois↑
7,0,cf_7,,,,,,1,37.07,,,,2,False,0.06,0.09,"NO: bmi↑, paccnois↑"
8,0,cf_8,5,,,,,,29.89,,,,2,False,0.06,0.06,"NO: bmi↑, etfruit↑"
9,0,cf_9,,,,,,,33.65,,,,1,False,0.06,0.22,NO: bmi↑


### is valid

In [18]:
# --- Select exactly one CF per query_index (random valid without violations) ---

# Split baseline and CF rows
df_xin = batch_cfcheck_df[batch_cfcheck_df["cf_id"] == "xin"]
df_cf  = batch_cfcheck_df[batch_cfcheck_df["cf_id"] != "xin"]

def select_random_cf(group):
    """Select one random CF that is valid and has no violations (Expected is empty)."""
    # First try: valid AND no violations
    good_cfs = group[(group["valid"] == True) & (group["Expected"] == "")]
    if len(good_cfs) > 0:
        return good_cfs.sample(n=1, random_state=42)

    # Fallback: any valid CF
    valid_cfs = group[group["valid"] == True]
    if len(valid_cfs) > 0:
        return valid_cfs.sample(n=1, random_state=42)

    # Last resort: first CF
    return group.iloc[[0]]

# Select one random CF per query_index
df_cf_selected = df_cf.groupby("query_index", group_keys=False).apply(select_random_cf).reset_index(drop=True)

# Combine baseline + selected CF
batch_cfcheck_df = pd.concat([df_xin, df_cf_selected], ignore_index=True)

# Ensure xin appears before CF for each query_index
batch_cfcheck_df["is_xin"] = (batch_cfcheck_df["cf_id"] == "xin").astype(int)
batch_cfcheck_df = (
    batch_cfcheck_df
    .sort_values(["query_index", "is_xin"], ascending=[True, False])
    .drop(columns="is_xin")
)
batch_cfcheck_df

/tmp/ipykernel_10590/2325069138.py:23: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_cf_selected = df_cf.groupby("query_index", group_keys=False).apply(select_random_cf).reset_index(drop=True)


,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,Gower,Nchanged,valid,risk_before,predicted_risk_after,Expected
0,0,xin,4,3,3,4,2,0,18.99,0,0.34,,,,,,
9,0,cf_5,,4,1,,,,,,,,2,True,0.06,0.03,"NO: cgtsmok↓, eatveg↑"
1,1,xin,3,4,1,2,3,0,22.38,0,0.33,,,,,,
10,1,cf_8,,,,5,,,,,,,1,True,0.15,0.07,
2,2,xin,5,3,1,1,4,0,22.69,7,0.32,,,,,,
11,2,cf_8,,,,,1,1,,,,,2,True,0.16,0.02,NO: paccnois↑
3,3,xin,3,3,6,6,2,0,24.34,1,0.34,,,,,,
12,3,cf_1,,,,,,,17.96,4,,,2,False,0.02,0.19,
4,4,xin,5,4,2,7,1,0,21.26,3,0.33,,,,,,
13,4,cf_5,4,2,,,,,,,,,2,True,0.03,0.01,


In [19]:
batch_cfcheck_df["valid"] = batch_cfcheck_df["valid"].replace(
    {
        False : "No",
        True : "Yes",
    }
)